# Coleta das Variáveis

## Distância da capital regional mais próxima (nível 5 ou superior)

### Coleta das Coordenadas

Amostra

In [ ]:
from pathlib import Path
import unicodedata

import pandas as pd

base = Path.cwd().parent / 'dados'
a = pd.read_csv(base / 'amostra.csv')
a.columns=['ID','Municipio','UF']
c = pd.read_csv(base / 'latitude-longitude-cidades.csv', sep=';')

norm=lambda v: '' if pd.isna(v) else ' '.join(unicodedata.normalize('NFKD', str(v).strip().casefold()).encode('ascii','ignore').decode('ascii').split())

a['m'] = a['Municipio'].map(norm)
a['u'] = a['UF'].map(norm)
c['m'] = c['municipio'].map(norm)
c['u'] = c['uf'].map(norm)

r = a.merge(c[['m','u','longitude','latitude']], on=['m','u'], how='left')
miss = r[r[['longitude','latitude']].isna().any(axis=1)]
print('miss', len(miss))

r = r.drop(columns=['m','u'])
r.to_csv(base / 'coordenadas_amostra.csv', index=False)
print(r.head().to_string(index=False))

miss 0
 ID      Municipio UF  longitude   latitude
407 Várzea da Roça BA -40.132829 -11.600463
337  Santa Bárbara BA -38.968085 -11.951490
538        Passira PE -35.581261  -7.997102
716   Missão Velha CE -39.143046  -7.235219
298          Piatã BA -41.770177 -13.146549


Capitais Regionais

In [17]:
from pathlib import Path
import unicodedata

import pandas as pd

base = Path.cwd().parent / 'dados'
coordenadas_path = base / "latitude-longitude-cidades.csv"

cidades = pd.DataFrame(
    [
        {"Municipio": "Salvador", "UF": "BA"},
        {"Municipio": "Feira de Santana", "UF": "BA"},
        {"Municipio": "Itabuna", "UF": "BA"},
        {"Municipio": "Vitória da Conquista", "UF": "BA"},
        {"Municipio": "Recife", "UF": "PE"},
        {"Municipio": "Caruaru", "UF": "PE"},
        {"Municipio": "Fortaleza", "UF": "CE"},
        {"Municipio": "Juazeiro do Norte", "UF": "CE"},
    ]
)
coordenadas = pd.read_csv(coordenadas_path, sep=";")


def normalizar_texto(valor: object) -> str:
    if pd.isna(valor):
        return ""
    texto = str(valor).strip().casefold()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caractere for caractere in texto if not unicodedata.combining(caractere))
    return " ".join(texto.split())


cidades["municipio_normalizado"] = cidades["Municipio"].apply(normalizar_texto)
cidades["uf_normalizada"] = cidades["UF"].apply(normalizar_texto)
coordenadas["municipio_normalizado"] = coordenadas["municipio"].apply(normalizar_texto)
coordenadas["uf_normalizada"] = coordenadas["uf"].apply(normalizar_texto)

resultado = cidades.merge(
    coordenadas[["municipio_normalizado", "uf_normalizada", "longitude", "latitude"]],
    on=["municipio_normalizado", "uf_normalizada"],
    how="left",
)

sem_coordenadas = resultado[resultado[["longitude", "latitude"]].isna().any(axis=1)]
if not sem_coordenadas.empty:
    print("Aviso: algumas cidades não foram encontradas no arquivo de coordenadas:")
    display(sem_coordenadas[["Municipio", "UF"]])

resultado = resultado.drop(columns=["municipio_normalizado", "uf_normalizada"])
resultado.to_csv(base / "coordenadas_crs.csv", index=False)

display(resultado)

,Municipio,UF,longitude,latitude
0,Salvador,BA,-38.501068,-12.971780
1,Feira de Santana,BA,-38.966293,-12.266429
2,Itabuna,BA,-39.278056,-14.787573
3,Vitória da Conquista,BA,-40.844159,-14.861466
4,Recife,PE,-34.877065,-8.046658
5,Caruaru,PE,-35.969863,-8.284547
6,Fortaleza,CE,-38.542298,-3.716638
7,Juazeiro do Norte,CE,-39.307593,-7.196207


### Cálculo da Distância

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np

base = Path.cwd().parent / 'dados'
amostra = pd.read_csv(base / 'amostra.csv')
coor_amostra = pd.read_csv(base / 'coordenadas_amostra.csv')
capitais = pd.read_csv(base / 'coordenadas_crs.csv')

df = amostra.merge(coor_amostra[['ID','longitude','latitude']], on='ID', how='left')
if df[['longitude','latitude']].isna().any(axis=1).any():
    print('Aviso: algumas cidades da amostra não têm coordenadas.')

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    R = 6371.0
    return R * c

# preparar arrays das capitais
cap_lons = capitais['longitude'].to_numpy()
cap_lats = capitais['latitude'].to_numpy()
cap_names = capitais['Municipio'].to_numpy()

nearest_names = []
nearest_dists = []
for lon, lat in zip(df['longitude'], df['latitude']):
    if pd.isna(lon) or pd.isna(lat):
        nearest_names.append(np.nan)
        nearest_dists.append(np.nan)
        continue
    dists = haversine(lon, lat, cap_lons, cap_lats)
    idx = np.nanargmin(dists)
    nearest_names.append(cap_names[idx])
    nearest_dists.append(float(dists[idx]))

df['distancia_capital_regional_km'] = np.round(nearest_dists, 3)
df.drop(columns=['longitude','latitude'], inplace=True)

df.to_csv(base / 'amostra_v1.csv', index=False)

display(df.head())

,ID,Município,UF,distancia_capital_regional_km
0,407,Várzea da Roça,BA,146.934
1,337,Santa Bárbara,BA,35.020
2,538,Passira,PE,53.398
3,716,Missão Velha,CE,18.663
4,298,Piatã,BA,215.275


## Áreas Verdes, Focos de Calor e Emissão de CO2

In [2]:
from pathlib import Path
import unicodedata
import pandas as pd
import numpy as np

base = Path.cwd().parent / 'dados'
# leitura da amostra previamente gerada
amostra = pd.read_csv(base / 'amostra_v1.csv')
# leitura do arquivo IPS (separador ; e encoding latin1 para acentos)
ips = pd.read_csv(base / 'ips-2025.CSV', sep=';', encoding='latin1')

def normalize_text(v: object) -> str:
    if pd.isna(v):
        return ''
    s = str(v).strip().casefold()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    return ' '.join(s.split())

# normalizar município e UF na amostra
amostra['mun_norm'] = amostra['Município'].map(normalize_text)
amostra['uf_norm'] = amostra['UF'].map(normalize_text)

# detectar colunas de município/UF no IPS (nomes podem ter problemas de encoding)
mun_col = None
uf_col = None
for c in ips.columns:
    cl = c.casefold()
    if 'munic' in cl and mun_col is None:
        mun_col = c
    if 'uf' == cl or cl == 'uf' or (cl == 'estado' and uf_col is None):
        uf_col = c
    elif 'uf' in cl and uf_col is None:
        uf_col = c

if mun_col is None or uf_col is None:
    raise ValueError('Não consegui localizar as colunas de município/UF no arquivo IPS')

ips['mun_norm'] = ips[mun_col].map(normalize_text)
ips['uf_norm'] = ips[uf_col].map(normalize_text)

# localizar colunas de interesse (áreas verdes, focos de calor, emissões de CO2) por padrões de texto
def find_col(patterns):
    for p in patterns:
        for c in ips.columns:
            if p in c.casefold():
                return c
    return None

verde_col = find_col(['verde', 'areas verdes', 'áreas verdes', 'areas verdes urbanas', 'áreas verdes urbanas'])
focos_col = find_col(['focos de calor', 'focos', 'calor'])
emiss_col = find_col(['emiss', 'co2', 'co2e', 'co?e', 'co'])

print('Colunas encontradas no IPS ->', 'verde:', verde_col, '| focos:', focos_col, '| emissões:', emiss_col)

# construir lista de colunas a trazer do IPS (somente as não-None)
cols_to_take = ['mun_norm','uf_norm']
for c in (verde_col, focos_col, emiss_col):
    if c is not None:
        cols_to_take.append(c)

ips_sel = ips[cols_to_take].copy()

# fazer merge
merged = amostra.merge(ips_sel, on=['mun_norm','uf_norm'], how='left')

# normalizar nomes de colunas resultantes e garantir colunas finais previsíveis
merged['areas_verdes_urbanas'] = merged[verde_col] if verde_col in merged.columns else np.nan
merged['focos_calor'] = merged[focos_col] if focos_col in merged.columns else np.nan
merged['emissao_co2_percapita'] = merged[emiss_col] if emiss_col in merged.columns else np.nan

# remover colunas auxiliares
drop_cols = ['mun_norm','uf_norm']
for c in (verde_col, focos_col, emiss_col):
    if c is not None and c in merged.columns:
        drop_cols.append(c)

merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])

# salvar nova amostra
merged.to_csv(base / 'amostra_v2.csv', index=False)

display(merged.head())

Colunas encontradas no IPS -> verde: Áreas Verdes Urbanas | focos: Focos de Calor | emissões: Emissões de CO?e por Habitante


,ID,Município,UF,distancia_capital_regional_km,areas_verdes_urbanas,focos_calor,emissao_co2_percapita
0,407,Várzea da Roça,BA,146.934,"1,16","2,17","5,11"
1,337,Santa Bárbara,BA,35.020,"2,41","0,48","2,88"
2,538,Passira,PE,53.398,"4,32",0,"2,83"
3,716,Missão Velha,CE,18.663,"1,28","13,58","4,94"
4,298,Piatã,BA,215.275,"5,67","4,48","9,53"
